# 02 — Exploratory Data Analysis: Diabetes Dataset

This notebook performs a comprehensive EDA on the PIMA Indians Diabetes dataset (768 rows, 9 columns).

**Target:** `Outcome` (0 = No Diabetes, 1 = Diabetes)

**Note:** Several features (Glucose, BloodPressure, SkinThickness, Insulin, BMI) use 0 as a placeholder for missing values.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASETS, RAW_DATA_DIR, SEED
from src.data.loader import load_raw_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Data

In [ ]:
df = load_raw_dataset('diabetes')
print(f"Shape: {df.shape}")
df.head()

## 2. Basic Info and Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Check for Zero-Encoded Missing Values

In this dataset, 0 is biologically implausible for several features and likely represents missing data.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Zero counts (likely missing):")
for col in zero_cols:
    n_zeros = (df[col] == 0).sum()
    print(f"  {col}: {n_zeros} ({n_zeros / len(df) * 100:.1f}%)")

## 4. Missing Values Heatmap

We replace 0 values in biologically implausible columns with NaN to visualize missingness.

In [ ]:
df_missing = df.copy()
for col in zero_cols:
    df_missing[col] = df_missing[col].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df_missing.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap — Diabetes Dataset (0 treated as NaN)')
plt.tight_layout()
plt.show()

## 5. Class Balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Outcome'].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
ax.set_ylabel('Count')
ax.set_title('Class Distribution — Diabetes')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Diabetes Dataset')
plt.tight_layout()
plt.show()

## 7. Feature Distributions (KDE) Split by Outcome

In [ ]:
numeric_features = DATASETS['diabetes']['numeric_features']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    for label in [0, 1]:
        subset = df[df['Outcome'] == label][col].dropna()
        sns.kdeplot(subset, ax=ax, label=f'Outcome={label}', fill=True, alpha=0.4)
    ax.set_title(f'{col} Distribution by Outcome')
    ax.legend()

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('KDE Distributions of Numeric Features — Diabetes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Box Plots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    sns.boxplot(x='Outcome', y=col, data=df, ax=ax)
    ax.set_title(f'{col} — Outliers')

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots of Numeric Features — Diabetes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Pair Plot for Top Correlated Features

In [ ]:
top_features = corr['Outcome'].drop('Outcome').abs().sort_values(ascending=False).head(4).index.tolist()
print(f"Top correlated features: {top_features}")

pair_df = df[top_features + ['Outcome']].dropna()
g = sns.pairplot(pair_df, hue='Outcome', diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.5})
g.figure.suptitle('Pair Plot — Top Correlated Features with Outcome', y=1.02)
plt.show()

## Summary

Key findings from this EDA:
- The dataset has 768 samples with a class imbalance (roughly 65% negative, 35% positive).
- Insulin and SkinThickness have a high proportion of zero values (likely missing).
- Glucose is the strongest predictor of diabetes outcome.
- BMI, Age, and Pregnancies also show meaningful separation between classes.
- Outliers are present in Insulin and some other features.